# AmsiNet - Step 18: example figures for the paper (CPU, a few minutes)
Makes two figures from **your own word crops** (not copied from other papers):
* **Figure A - what the data looks like:** a grid of random test word crops for each of the ten scripts.
* **Figure B - what context does:** lines in which the word agent alone makes errors that the cascade corrects, shown as crops with the true script, the word agent's guess and the final decision.

Needs the Step 2 crops (`crops224.tar` on Drive) and the Step 3a agents. Saves PNG and PDF to `MyDrive/amsinet/figures/`. Download them and send them to Claude.

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
import json, numpy as np, pandas as pd
from scipy.special import softmax
from scipy import stats
ROOT = '/content/drive/MyDrive/amsinet'
CFG = dict(SEEDS=[0, 1, 2, 3, 4], VARIANTS=['word', 'word_moe'], EPS=1e-4, KEEP=0.90)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
SCRIPTS, INDEX = D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); REGDF = pd.read_csv(f'{ROOT}/data/regions_meta.csv')
eff = pd.read_csv(f'{ROOT}/agents/agent_efficiency_cpu.csv').set_index('agent')['GFLOPs']
flat_df = pd.read_csv(f'{ROOT}/results/backbone_efficiency.csv'); FLAT = float(flat_df[flat_df.agent == 'Flat baseline']['GFLOPs'].iloc[0])
missing = [f'{t}_seed{s}' for s in CFG['SEEDS'] for t in ('block', 'line', 'word', 'word_moe') if not os.path.exists(f'{ROOT}/agents/{t}_seed{s}_test_probs.npy')]
print('missing agent files:', missing or 'none')

In [ ]:
# 2. Helpers (same definitions as Steps 6 and 8)
def load(tag, sp):
    P = np.load(f'{ROOT}/agents/{tag}_{sp}_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv'); return P, pa.path.tolist()
def region_lookup(level, tag, sp):
    P, paths = load(tag, sp); pos = {p: i for i, p in enumerate(paths)}
    d = REGDF[(REGDF.level == level) & (REGDF.split == sp)]
    return {(r.image_id, r.rid): P[pos[r.path]] for r in d.itertuples() if r.path in pos}
def table(sp, wtag, ltag, btag):
    d = META[META.split == sp].reset_index(drop=True); n = len(d)
    Pw, pw = load(wtag, sp); assert pw == d.path.tolist(), 'word probabilities are not aligned with crops_meta.csv'
    logP = np.log(np.clip(Pw, CFG['EPS'], 1)); lid, bid = [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
    img = d.image_id.values; lkey = img.astype(str) + '|l' + np.array(lid).astype(str); bkey = img.astype(str) + '|b' + np.array(bid).astype(str)
    lg, lu = pd.factorize(lkey); bg, bu = pd.factorize(bkey)
    def loo(g, ng):
        S = np.zeros((ng, C)); cnt = np.zeros(ng); np.add.at(S, g, logP); np.add.at(cnt, g, 1); o = cnt[g] - 1
        F = np.zeros((n, C)); m = o > 0; F[m] = (S[g][m] - logP[m]) / o[m][:, None]; return F
    def reg(level, tag, key):
        L = region_lookup(level, tag, sp); F = np.zeros((n, C))
        for i, (iid, kk) in enumerate(zip(img, key)):
            v = L.get((iid, f'{level[0]}{kk}'))
            if v is not None: F[i] = np.log(np.clip(v, CFG['EPS'], 1))
        return F
    ui, inv = np.unique(img, return_inverse=True); l_img = np.zeros(len(lu), int); l_img[lg] = inv; b_img = np.zeros(len(bu), int); b_img[bg] = inv
    return dict(y=np.array([SIDX[s] for s in d.script]), word=logP, line_ctx=loo(lg, len(lu)), block_ctx=loo(bg, len(bu)),
                line_agent=reg('line', ltag, lid), block_agent=reg('block', btag, bid),
                img=img, inv=inv, n_img=len(ui), lg=lg, bg=bg, nl=len(lu), nb=len(bu), l_img=l_img, b_img=b_img)
def confmat(y, p): return np.bincount(y * C + p, minlength=C * C).reshape(C, C)
def f1_from(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
def metr(y, p): cm = confmat(y, p); return dict(acc=100 * float(np.trace(cm) / cm.sum()), f1=100 * float(f1_from(cm).mean()))
def unc(s): p = softmax(s, axis=1); return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1) / np.log(C)
KEYS = ['line_ctx', 'block_ctx', 'line_agent', 'block_agent']; GRID_W = [0, .1, .25, .5, 1, 1.5, 2, 3]
def cascade(T, tau, W):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']; u = unc(s)
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], u > tau); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], u > tau); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def cost(T, li, bi, GF):
    nw = np.bincount(T['inv'], minlength=T['n_img']).astype(float); nl = np.bincount(T['l_img'][li], minlength=T['n_img']).astype(float); nb = np.bincount(T['b_img'][bi], minlength=T['n_img']).astype(float)
    return float((nw * GF['word'] + nl * GF['line'] + nb * GF['block']).mean())
TAUS = [0, .02, .05, .1, .15, .2, .3, .4, .5, .6, .7, .8, .9, 1.01]

## 3. Predictions of the final system (seed 0, MoE head)

In [ ]:
def run(seed, variant, keep_preds=False):
    wtag, ltag, btag = f'{variant}_seed{seed}', f'line_seed{seed}', f'block_seed{seed}'
    GF = dict(word=float(eff[variant]), line=float(eff['line']), block=float(eff['block']))
    TV, TT = table('val', wtag, ltag, btag), table('test', wtag, ltag, btag)
    def full(T, W): return T['word'] + sum(W[k] * T[k] for k in KEYS)
    vf1 = lambda W: metr(TV['y'], full(TV, W).argmax(1))['f1']
    W = {k: 0.0 for k in KEYS}
    for _ in range(2):
        for k in KEYS: W[k] = max(GRID_W, key=lambda g: vf1({**W, k: g}))
    cv = {t: (lambda r: (metr(TV['y'], r[0].argmax(1))['f1'], cost(TV, r[1], r[2], GF)))(cascade(TV, t, W)) for t in TAUS}
    f_full, f_ctx = cv[0][0], cv[1.01][0]; target = f_ctx + CFG['KEEP'] * (f_full - f_ctx)
    ok = [t for t in TAUS if cv[t][0] >= target]; tau = min(ok, key=lambda t: cv[t][1]) if ok else 0.0
    out = dict(seed=seed, variant=variant, tau_star=tau, **{f'w_{k}': W[k] for k in KEYS})
    y = TT['y']; res = {}
    for name, (s, li, bi) in dict(word=(TT['word'], np.zeros(TT['nl'], bool), np.zeros(TT['nb'], bool)), context=cascade(TT, 1.01, W), cascade=cascade(TT, tau, W), full=cascade(TT, 0, W)).items():
        m = metr(y, s.argmax(1)); out[f'{name}_acc'], out[f'{name}_f1'], out[f'{name}_gflops'] = m['acc'], m['f1'], cost(TT, li, bi, GF); res[name] = s.argmax(1)
        if name == 'cascade': out['lines_pct'], out['blocks_pct'] = 100 * li.mean(), 100 * bi.mean()
    # mixed-script lines: lines whose words have more than one true script
    first = np.full(TT['nl'], -1); mixed = np.zeros(TT['nl'], bool)
    for g, yy in zip(TT['lg'], y):
        if first[g] < 0: first[g] = yy
        elif first[g] != yy: mixed[g] = True
    mw = mixed[TT['lg']]; out['mixed_line_pct'] = 100 * mixed.mean(); out['mixed_word_pct'] = 100 * mw.mean()
    for name in ('word', 'context', 'cascade', 'full'):
        out[f'{name}_acc_mixed'] = 100 * float((res[name][mw] == y[mw]).mean()); out[f'{name}_acc_single'] = 100 * float((res[name][~mw] == y[~mw]).mean())
    return (out, dict(y=y, **res, mw=mw, conf=np.exp(TT['word']).max(1), nline=np.bincount(TT['lg'])[TT['lg']], paths=META[META.split == 'test'].path.tolist())) if keep_preds else (out, None)

In [ ]:
import subprocess, os
LOCAL = '/content/bstd'; CROPS, CROPS_TAR = f'{LOCAL}/crops224', f'{ROOT}/data/crops224.tar'
os.makedirs(f'{ROOT}/figures', exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
if not os.path.exists(META.path.iloc[0]) and os.path.exists(CROPS_TAR): print('restoring crops'); subprocess.run(['tar', '-xf', CROPS_TAR, '-C', LOCAL], check=True)
assert os.path.exists(META.path.iloc[0]), 'crops not found: ' + META.path.iloc[0]
o, P = run(0, 'word_moe', keep_preds=True)
TE = META[META.split == 'test'].reset_index(drop=True); assert TE.path.tolist() == P['paths']
y, pw, pc = P['y'], P['word'], P['cascade']
TE['line'] = [IDX[i]['words'][k].get('line', -1) for i, k in zip(TE.image_id, TE.word_idx)]
TE['lkey'] = TE.image_id.astype(str) + '|' + TE.line.astype(str)
print('word agent alone acc %.2f | cascade acc %.2f' % (100 * (pw == y).mean(), 100 * (pc == y).mean()))

## Figure A: sample crops per script

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
rs = np.random.RandomState(7); NCOL = 8
order = sorted(range(C), key=lambda s: -int((y == s).sum()))
fig, ax = plt.subplots(C, NCOL, figsize=(NCOL * 1.15, C * 1.0))
for r, s in enumerate(order):
    idx = np.where(y == s)[0]; idx = idx[[len(TE.path[i]) > 0 for i in idx]]
    # prefer crops that are not tiny: use aspect ratio and file size as a rough filter
    pick = rs.choice(idx, NCOL, replace=False)
    for c, i in enumerate(pick):
        a = ax[r, c]; a.imshow(Image.open(TE.path[i]).convert('RGB')); a.set_xticks([]); a.set_yticks([])
        for sp in a.spines.values(): sp.set_visible(False)
        if c == 0: a.set_ylabel(SCRIPTS[s], rotation=0, ha='right', va='center', fontsize=8)
fig.subplots_adjust(left=0.2, right=0.995, top=0.995, bottom=0.005, wspace=0.04, hspace=0.06)
fig.savefig(f'{ROOT}/figures/FigA_sample_word_crops.png', dpi=250); fig.savefig(f'{ROOT}/figures/FigA_sample_word_crops.pdf'); plt.show()

## Figure B: lines where context corrects the word agent

In [ ]:
# lines with at least 4 words, one script only, where the alone-prediction has errors that the cascade fixes and the cascade has none
cand = []
for lk, g in TE.groupby('lkey'):
    ii = g.index.values
    if len(ii) < 4 or len(ii) > 8 or len(set(y[ii])) != 1: continue
    nwrong_alone, nwrong_casc = int((pw[ii] != y[ii]).sum()), int((pc[ii] != y[ii]).sum())
    if nwrong_alone >= 1 and nwrong_casc == 0: cand.append((nwrong_alone, lk, ii))
print(len(cand), 'candidate lines')
# choose a variety of scripts, preferring non-Latin lines with the most corrections
cand.sort(key=lambda t: -t[0]); chosen, seen = [], set()
for nw, lk, ii in cand:
    s = y[ii[0]]
    if s in seen: continue
    chosen.append((lk, ii)); seen.add(s)
    if len(chosen) == 4: break
rows = len(chosen); wmax = max(len(ii) for _, ii in chosen)
fig, ax = plt.subplots(rows * 2, wmax, figsize=(wmax * 1.3, rows * 1.9), squeeze=False)
for a in ax.ravel(): a.axis('off')
SH = lambda n: {'Bengali-Assamese': 'Beng.-Assam.'}.get(n, n)
for r, (lk, ii) in enumerate(chosen):
    for c, i in enumerate(ii):
        a = ax[2 * r, c]; a.imshow(Image.open(TE.path[i]).convert('RGB')); a.axis('off')
        ok_alone, ok_casc = pw[i] == y[i], pc[i] == y[i]
        t = ax[2 * r + 1, c]
        t.text(0.5, 0.95, 'alone: ' + SH(SCRIPTS[pw[i]]), ha='center', va='top', fontsize=6.3, color='#1b7837' if ok_alone else '#b2182b', transform=t.transAxes)
        t.text(0.5, 0.45, 'cascade: ' + SH(SCRIPTS[pc[i]]), ha='center', va='top', fontsize=6.3, color='#1b7837' if ok_casc else '#b2182b', transform=t.transAxes)
    ax[2 * r, 0].text(-0.05, 0.5, 'true: ' + SCRIPTS[y[ii[0]]], ha='right', va='center', fontsize=7.5, transform=ax[2 * r, 0].transAxes)
fig.subplots_adjust(left=0.17, right=0.995, top=0.995, bottom=0.005, wspace=0.05, hspace=0.12)
fig.savefig(f'{ROOT}/figures/FigB_context_corrections.png', dpi=250); fig.savefig(f'{ROOT}/figures/FigB_context_corrections.pdf'); plt.show()
print('saved to', f'{ROOT}/figures/')